[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Array_Processing.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Array Processing & Beamforming

Filtering in **space**: with several microphones/antennas, you can point a 'listening beam' at a direction, null an interferer, and locate sources — all with the same linear algebra as temporal filtering. Three sessions from array geometry to MUSIC.

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4 (eigen/subspaces) for Session 3.
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S3 for MVDR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Uniform linear array (ULA): M sensors, half-wavelength spacing
M = 8
def steering(theta_deg):
    """Array response to a far-field narrowband source at angle theta (broadside = 0°)."""
    theta = np.deg2rad(theta_deg)
    return np.exp(1j * np.pi * np.arange(M) * np.sin(theta))   # d = λ/2

---
### 🕐 Session 1 of 3 — *The Array Manifold* (~35 min)
**Goal:** understand why direction becomes a phase pattern across sensors.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S4. &nbsp; **Feeds into:** Session 2 (beamforming).

---

## 2. Direction Is a Spatial Frequency

💡 **Intuition.** A far-field wavefront hits each sensor at a slightly different time; for a narrowband signal, delay ≈ phase shift. Across a uniform line of sensors the phases advance *linearly* — direction $\theta$ shows up as a **spatial sinusoid** with frequency $\propto \sin\theta$. Everything you know about temporal frequencies transfers verbatim: sensors ↔ samples, aperture ↔ record length, beamwidth ↔ resolution, and spacing $> \lambda/2$ ⇒ *spatial aliasing* (grating lobes) — Nyquist in space.

In [ ]:
# The steering vector IS a sampled sinusoid — see it

# YOUR CODE HERE


**What just happened.** Three steering vectors, plotted across the eight sensors, and each is a **sampled sinusoid**. At $\theta = 0°$ the wavefront arrives at every sensor simultaneously, so the phase is constant — spatial DC. At 20° the phases advance slowly across the array; at 60° they advance rapidly. Direction has become frequency, and the sensor index is playing the role of time.

That is the entire conceptual move of the workshop. Everything this room learned in [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) transfers with a change of variable names: sensors are samples, the array aperture is the record length, beamwidth is frequency resolution, and element spacing is the sampling interval. Beamforming will turn out to be a DFT evaluated at one spatial frequency, and direction finding a spectral estimation problem. Nothing new is being invented — a familiar toolkit is being pointed at a different axis.

**Two consequences worth reading off these plots.**

The spatial frequency is proportional to $\sin\theta$, not to $\theta$. Compare the 0° and 20° panels against the 20° and 60° panels: the same 20-degree step changes the pattern much less near broadside than near endfire... and in fact rather *more* between 0° and 20° than between 40° and 60°, since $\sin$ flattens as it approaches 1. Either way the mapping is nonlinear, so angular resolution is not uniform across the field of view — arrays see most sharply at broadside and worst at endfire, which is a real siting consideration.

And because $\sin\theta$ is bounded by 1, the spatial frequency is bounded too. Half-wavelength spacing places that maximum exactly at the spatial Nyquist limit. Space the elements further apart and distinct directions start producing identical phase patterns — **grating lobes**, which are aliasing in space, indistinguishable from the real thing and unfixable after the fact. The $\lambda/2$ in the code is the sampling theorem, enforced.

Note finally that only the real part is plotted. The steering vector is complex, and the imaginary part carries the other half of the phase information — which is why a real-valued array cannot distinguish $+\theta$ from $-\theta$ without additional structure.

---
### 🕐 Session 2 of 3 — *Delay-and-Sum & MVDR* (~40 min)
**Goal:** steer beams; then let the data place nulls on interferers automatically.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (subspace methods).

---

## 3. Beamforming

💡 **Intuition.** **Delay-and-sum**: phase-align the sensors toward $\theta_0$ and add — signals from $\theta_0$ stack coherently ($M\times$ amplitude), others partially cancel. It's a matched filter in space, and like all matched filters it's optimal in white noise but naive about *structured* interference. **MVDR (Capon)** fixes that: minimize output power subject to unit gain at $\theta_0$ — $\mathbf{w} = \frac{R^{-1} \mathbf{a}}{\mathbf{a}^H R^{-1} \mathbf{a}}$ — a [Lagrange problem](../Intro_Math/Optimization/Optimization.ipynb) whose solution *automatically digs nulls* wherever the covariance says energy is coming from.

In [ ]:
# Scene: desired source at 0°, LOUD interferer at 40°, noise

# YOUR CODE HERE


**What just happened.** Output SINR of **7.0 dB** for delay-and-sum against **18.4 dB** for MVDR — an 11.4 dB improvement from the same eight sensors and the same data. The interferer at 40° is 3× the desired amplitude, about 9.5 dB stronger in power, and the two beamformers deal with it completely differently.

**Read the patterns, because they explain the numbers.** Delay-and-sum has the beam shape you would predict from [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) alone: a main lobe at 0° and a fixed sidelobe structure that is the transform of a rectangular window. Crucially that pattern was determined *before any data arrived* — it depends only on geometry. Wherever the interferer happens to land, it is attenuated by whatever the sidelobe level is there and no more, which here leaves it dominating the output.

MVDR's pattern keeps unit gain at 0° and drives a deep, narrow null down at exactly 40°. Nothing in the code mentions 40°. The interferer's direction is never estimated, never passed as a parameter, never searched for. It emerges from $R$ alone: the constraint $w^H a(\theta_0) = 1$ pins the desired direction, and minimising $w^H R w$ then suppresses everything else it can — and the only large "everything else" is at 40°. Nulls are what optimality *does* when there is structured interference to reject.

**Why the SINR gap is smaller than the null is deep.** The null is 40+ dB, but SINR only improves by 11.4 dB, because once the interferer is removed the output is limited by the remaining white noise — which MVDR cannot beat, since suppressing spatially white noise is exactly what delay-and-sum already does optimally. MVDR wins on the *structured* component and ties on the unstructured one.

**Now the honest framing, because MVDR is famously brittle.** This demo is the best case in every respect: 4000 snapshots to estimate an $8\times 8$ covariance, exact steering vectors, and sources that are independent. Change any of these and it degrades sharply.

- **Steering vector mismatch.** The constraint protects the direction you *claim* is the signal. Get $\theta_0$ slightly wrong — through calibration error, element position error, or an unmodelled array response — and MVDR treats the real signal as interference and nulls it. It can then perform *worse* than delay-and-sum, which has no such failure mode. This is the classic "signal cancellation" problem.
- **Finite snapshots.** $R$ here is estimated, not known. With fewer snapshots than sensors it is singular and `solve` fails; even with a few times $M$ it is poorly conditioned. Production systems add diagonal loading, $R + \epsilon I$, which trades a little null depth for robustness.
- **Coherent sources.** Multipath produces correlated copies of the same signal, which breaks the covariance structure the method assumes; spatial smoothing is the standard repair.

The general shape is worth carrying away: adaptive methods buy large gains by exploiting structure in the data, and pay for it with sensitivity to whether that structure is really what you assumed. Session 3's MUSIC makes the same bargain, more aggressively.

---
### 🕐 Session 3 of 3 — *Subspace Methods: MUSIC* (~40 min)
**Goal:** use covariance eigenstructure to localize sources beyond the beamwidth limit.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4.

---

## 4. MUSIC

💡 **Intuition.** With $K$ sources, the covariance's top-$K$ eigenvectors span the **signal subspace** (where steering vectors of true directions live); the remaining eigenvectors span the orthogonal **noise subspace**. MUSIC scans directions and scores each by *how orthogonal its steering vector is to the noise subspace* — true directions produce near-zero projections and towering pseudo-spectrum peaks. This is [Linear Algebra S3](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)'s eigen-story paying rent: resolution beyond the classical beamwidth.

In [ ]:
# Two sources only 8° apart — closer than the array's beamwidth

# YOUR CODE HERE


**What just happened.** MUSIC recovered `[-3. 5.]` against a truth of `[-3, 5]` — both directions, exactly, from two sources separated by **8°** on an array whose classical beamwidth is about **14°**. The left panel shows what classical processing can do with the same data: a single blurred bump that gives no hint there are two sources at all.

That comparison is the result. The right panel is not a better-tuned version of the left one; the two panels answer different questions. The classical scan measures *power arriving from* each direction, and power measurement is limited by beamwidth — a 4-wavelength aperture cannot resolve finer, and no amount of processing changes the geometry. MUSIC instead measures *orthogonality to the noise subspace*, which is a geometric yes/no condition with no beamwidth attached.

**The mechanism, concretely.** With 2 sources and 8 sensors, $R$ has 2 large eigenvalues and 6 small ones. The 6 small eigenvectors span a noise subspace that is orthogonal to every true steering vector. Scanning $\theta$ and computing $1/\|E_n^H a(\theta)\|^2$ therefore produces a near-division-by-zero precisely at $-3°$ and $5°$, and nothing special anywhere else. The peaks are sharp because the projection collapses fast as $\theta$ approaches truth, not because the array suddenly acquired more aperture.

**Read the y-axis correctly — this is the most common misreading.** The pseudo-spectrum is *not* power. Its height is the reciprocal of a projection residual, so peak heights carry no physical meaning: a weak source with a clean subspace can peak higher than a strong one. MUSIC tells you *where* sources are, not how loud they are. If you need amplitudes, you estimate them separately once the directions are known.

**And the assumptions this rests on, which are substantial.** `K = 2` is *given* to the algorithm, not discovered — the line `En = evecs[:, :M-K]` needs to know how many sources exist. Get $K$ wrong and the method breaks in both directions: too small leaves signal energy inside the supposed noise subspace and buries the real peaks; too large consumes the noise subspace and invents spurious ones. Real systems estimate $K$ first from the eigenvalue spectrum (AIC, MDL), and that is its own difficult problem.

Beyond that: the sources here are *independent*, so the signal covariance has full rank $K$. Coherent sources — multipath, the normal case in a room or an urban channel — make it rank-deficient and MUSIC fails outright, needing spatial smoothing to repair. And we had 4000 snapshots at good SNR; the sharpness of these peaks reflects a clean eigen-decomposition, not intrinsic precision. Lower the SNR or shorten the record and the peaks broaden, then merge. A razor-sharp peak in the wrong place is still wrong, and this plot's confidence is not itself evidence.

**The bargain, stated once.** Classical beamforming assumes almost nothing and is limited by physics. Subspace methods assume a specific model — known $K$, independent sources, accurate steering vectors — and in exchange beat the physical limit. That is the same trade MVDR made in Session 2, pushed further: more structure assumed, more performance when the assumption holds, more catastrophic failure when it does not.

## 5. Conclusion

Direction = spatial frequency; delay-and-sum = spatial matched filter; MVDR = constrained optimization that nulls interference by itself; MUSIC = eigen-subspace geometry beating the beamwidth. Space is just another axis to filter.

---
## Where next

- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) — the covariance machinery underneath.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — arrays of real antennas.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — microphone arrays in your smart speaker run exactly this.